# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

# Projektpfade
RAW_PATH = Path("../data/raw/brazilian-ecommerce")
DB_PATH = Path("../data/processed/olist.duckdb")

# Verbindung zur DuckDB als Datei (persistente DB statt in-memory)
con = duckdb.connect(DB_PATH.as_posix())

# Hilfsfunktion für SQL-Abfragen
def sql(query: str):
    return con.execute(query).df()

# Alle CSV-Dateien in DuckDB als Tabellen speichern (persistent in olist.duckdb)
for file in RAW_PATH.glob("*.csv"):
    table_name = file.stem.replace("olist_", "").replace("_dataset", "")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{file.as_posix()}');
    """)

# Alle Tabellen anzeigen
sql("SHOW TABLES")

,name
0,customers
1,geolocation
2,order_items
3,order_payments
4,order_reviews
5,orders
6,product_category_name_translation
7,products
8,sellers


## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [2]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [3]:
df_rfm_eda

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
0,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,47813,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,118.70,22.76,1
1,9b18f3fc296990b97854e351334a32f6,b2cac0b16835dabf811b204127f58afa,carapicuiba,SP,06330,138849fd84dff2fb4ca70a0a34c4aa1c,delivered,2018-02-01 14:02:19,2018-02-03 02:53:07,39.47,13.37,1
2,bb2f5e670f7155dc622c57e4b31d0a69,31b8fa2573bde01af4737e8ed29c348b,sao paulo,SP,02346,a6aeb116d2cb5013eb8a94585b71ffef,delivered,2017-09-13 14:27:11,2017-09-13 14:44:39,50.00,9.34,1
3,f26a435864aebedff7f7c84f82ee229f,bb4d84a2b45b22ed710ac8c0dec63d1a,poa,SP,08552,b8801cccd8068de30112e4f49903d74a,delivered,2017-07-30 03:06:35,2017-07-30 03:25:08,19.99,7.78,1
4,803ac05904124294f8767894d6da532b,34c58672601f2c6d29db7efd1f6bf958,bonfinopolis de minas,MG,38650,bfe42c22ecbf90bc9f35cf591270b6a7,delivered,2018-01-27 22:04:34,2018-01-27 22:16:18,27.30,15.10,1
...,...,...,...,...,...,...,...,...,...,...,...,...
101565,59b9edf2b330ded2efc5d1112b45a660,6782d39234f06de1548c8e8514082d63,toledo,PR,85900,93843243cdd92e9fdfd9a1eb7429ddf1,delivered,2018-01-03 17:29:21,2018-01-03 17:46:53,24.99,15.10,1
101566,aa179ef3bea6245ec335cd5f2c65044c,d73238dc01a257f5439f611e3cdb199c,porto alegre,RS,91410,7c29f6fa9efcbf410b4bbb9362b9f7c9,delivered,2017-10-21 22:51:11,2017-10-22 18:35:04,27.40,11.96,1
101567,afe44c51b2ff47d26e457d10585bf8b4,1412a71cfa4f44a7e5473220f820b4d9,recife,PE,51010,c7387d86e09f2823f6062d8c2378ce02,delivered,2018-08-08 15:01:55,2018-08-08 15:10:23,45.00,19.22,1
101568,e806ed5d5fdf4be71b5405c1d8e39b8f,9c4c1712431d24464571f61635d5c6fe,cotia,SP,06724,78db0b29049b6795343e2a8ba2f48ff7,shipped,2017-11-09 22:29:00,2017-11-09 22:46:51,43.99,7.78,1


In [4]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04.180673,2018-01-01 12:06:22.988666,124.922151,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48.500000,40.800000,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20.500000,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31.500000,139.530000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,20.000000
std,NaN,NaN,189.479405,15.901388,0.474137


#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [5]:
df_rfm_eda.dtypes

customer_id                         object
customer_unique_id                  object
customer_city                       object
customer_state                      object
customer_zip_code_prefix            object
order_id                            object
order_status                        object
order_purchase_timestamp    datetime64[us]
order_approved_at           datetime64[us]
price                              float64
freight_value                      float64
product_count                        int64
dtype: object

In [6]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

customer_id                      category
customer_unique_id               category
customer_city                    category
customer_state                   category
customer_zip_code_prefix         category
order_id                         category
order_status                     category
order_purchase_timestamp    datetime64[s]
order_approved_at           datetime64[s]
price                             float32
freight_value                     float32
product_count                       int16
dtype: object

In [7]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101570,101556,101570.000000,101570.000000,101570.000000
mean,2018-01-01 00:42:04,2018-01-01 12:06:22,124.922150,20.140526,1.109087
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 08:05:38,2017-09-13 17:27:48,40.799999,13.160000,1.000000
50%,2018-01-19 16:57:45,2018-01-20 09:09:20,79.000000,16.340000,1.000000
75%,2018-05-04 23:26:44,2018-05-05 12:55:31,139.529995,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,189.479401,15.901388,0.474137


In [8]:
df_rfm_eda.isna().sum()

customer_id                  0
customer_unique_id           0
customer_city                0
customer_state               0
customer_zip_code_prefix     0
order_id                     0
order_status                 0
order_purchase_timestamp     0
order_approved_at           14
price                        0
freight_value                0
product_count                0
dtype: int64

In [9]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10452,07a2a7e0f63fd8cb757ed77d4245623c,79af1bbf230a2630487975aa5d7d6220,paraisopolis,MG,37660,51eb2eebd5d76a24625b31c33dd41449,delivered,2017-02-18 15:52:27,NaT,59.900002,17.160000,1
22980,a3d3c38e58b9d2dfb9207cab690b6310,5a4fa4919cbf2b049e72be460a380e5b,abaete,MG,35620,2eecb0d85f281280f79fa00f9cec1a95,delivered,2017-02-17 17:21:55,NaT,135.000000,19.230000,1
24209,0bf35cac6cc7327065da879e2d90fae8,c4c0011e639bdbcf26059ddc38bd3c18,varzea paulista,SP,13225,d77031d6a3c8a52f019764e68f211c69,delivered,2017-02-18 11:04:19,NaT,28.990000,10.960000,1
24740,d85919cb3c0529589c6fa617f5f43281,c094ac95fcd52f821809ec232a7a6956,sao vendelino,RS,95795,3c0b8706b065f9919d0505d3b3343881,delivered,2017-02-17 15:53:27,NaT,133.990005,23.200001,1
39867,4c1ccc74e00993733742a3c786dc3c1f,91efb7fcabc17925099dced52435837f,novo hamburgo,RS,93548,8a9adc69528e1001fc68dd0aaebbb54a,delivered,2017-02-18 12:45:31,NaT,379.000000,17.860001,1
49311,2127dc6603ac33544953ef05ec155771,8a9a08c7ca8900a200d83cf838a07e0b,cotia,SP,06708,e04abd8149ef81b95221e88f6ed9ab6a,delivered,2017-02-18 14:40:00,NaT,309.899994,39.110001,1
49640,684cb238dc5b5d6366244e0e0776b450,6ff8b0d7b35d5c945633b8d60165691b,santos,SP,11030,c1d4211b3dae76144deccd6c74144a88,delivered,2017-01-19 12:48:08,NaT,39.990002,14.520000,1
50272,f67cd1a215aae2a1074638bbd35a223a,bc1896dc77f49e6dec880445a9b443a3,rio de janeiro,RJ,21020,88083e8f64d95b932164187484d90212,delivered,2017-02-18 22:49:19,NaT,49.000000,14.520000,2
51178,1e101e0daffaddce8159d25a8e53f2b2,c8822fce1d0bfa7ddf0da24fff947172,macae,RJ,27945,12a95a3c06dbaec84bcfb0e2da5d228a,delivered,2017-02-17 13:05:55,NaT,79.989998,15.770000,1
57738,68d081753ad4fe22fc4d410a9eb1ca01,2e0a2166aa23da2472c6a60c4af6f7a6,sao paulo,SP,03573,d69e5d356402adc8cf17e08b5033acfb,delivered,2017-02-19 01:28:47,NaT,149.800003,13.630000,1


In [10]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

order_approved_at,False,True
order_status,,
approved,2,0
canceled,464,0
delivered,99341,14
invoiced,319,0
processing,304,0
shipped,1119,0
unavailable,7,0


In [11]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count


In [12]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

Gesamte Duplikate: 0


In [13]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable
order_id,,,,,,,
00010242fe8c5a6d1ba2dd792cb16214,0,0,1,0,0,0,0
00018f77f2f0320c557190d7a144bdd3,0,0,1,0,0,0,0
000229ec398224ef6ca0657da4fc703e,0,0,1,0,0,0,0
00024acbcdf0a6daa1e931b038114c75,0,0,1,0,0,0,0
00042b26cf59d7ce69dfabb4e55b4fd9,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...
fffc94f6ce00a00581880bf54a75a037,0,0,1,0,0,0,0
fffcd46ef2263f404302a634eb57f7eb,0,0,1,0,0,0,0
fffce4705a9662cd70adb13d4a31832d,0,0,1,0,0,0,0


In [14]:
test['sum_status']=test.sum(axis=1)

In [15]:
test.loc[test['sum_status']!=1, :] 

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
002f98c0f7efd42638ed6100ca699b42,0,0,2,0,0,0,0,2
005d9a5423d47281ac463a968b3936fb,0,0,2,0,0,0,0,2
00946f674d880be1f188abc10ad7cf46,0,0,2,0,0,0,0,2
0097f0545a302aafa32782f1734ff71c,0,0,2,0,0,0,0,2
00bcee890eba57a9767c7b5ca12d3a1b,0,0,2,0,0,0,0,2
...,...,...,...,...,...,...,...,...
ffb18bf111fa70edf316eb0390427986,0,0,2,0,0,0,0,2
ffb8f7de8940249a3221252818937ecb,0,0,3,0,0,0,0,3
ffb9a9cd00c74c11c24aa30b3d78e03b,0,0,3,0,0,0,0,3


####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [16]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [17]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

order_status,approved,canceled,delivered,invoiced,processing,shipped,unavailable,sum_status
order_id,,,,,,,,
ca3625898fbd48669d50701aba51cd5f,0,0,7,0,0,0,0,7
cf5c8d9f52807cb2d2f0a0ff54c478da,0,0,6,0,0,0,0,6
5a3b1c29a49756e75f1ef513383c0c12,0,0,6,0,0,0,0,6
b436eb981676e54c0bc9bcade0e079c4,0,0,5,0,0,0,0,5
bb82809ea3ca9f3edbe589b60e14e0cb,0,0,5,0,0,0,0,5
...,...,...,...,...,...,...,...,...
59b67c775c6a905fc4faac69ca74b5cb,0,0,2,0,0,0,0,2
59bccab4e9193a9229f7d1b73fcb47c3,0,0,2,0,0,0,0,2
59c0ed646a3b30d4054298988188486f,0,0,2,0,0,0,0,2


## Filterung nach der Bestellung mit den meisten Duplikaten

In [18]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

,customer_id,customer_unique_id,customer_city,customer_state,customer_zip_code_prefix,order_id,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
10767,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,309.000000,1.84,1
11950,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,56.000000,3.68,2
32913,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,33.900002,1.84,1
42547,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,95.900002,0.15,2
44162,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,109.900002,0.15,1
52119,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,63.700001,0.15,1
64688,0d861a5e4dd6a9079d89e1330848f0ab,c8ed31310fc440a3f8031b177f9842c3,ipua,SP,14610,ca3625898fbd48669d50701aba51cd5f,delivered,2018-08-12 02:11:20,2018-08-12 02:25:07,159.000000,3.67,2


In [19]:
df_rfm_eda.describe()

,order_purchase_timestamp,order_approved_at,price,freight_value,product_count
count,101099,101099,101099.000000,101099.000000,101099.000000
mean,2018-01-01 04:49:34,2018-01-01 15:09:29,124.649025,20.140503,1.108824
min,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.000000
25%,2017-09-13 11:58:02,2017-09-13 20:10:21,40.799999,13.180000,1.000000
50%,2018-01-19 16:33:57,2018-01-20 09:08:37,79.000000,16.350000,1.000000
75%,2018-05-05 07:49:38,2018-05-05 14:13:51,139.000000,21.260000,1.000000
max,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993,20.000000
std,NaN,NaN,188.526642,15.891350,0.473023


In [20]:
con.execute("CREATE TABLE customer_rfm AS SELECT * FROM df_rfm_eda")

### EDA für zweite Kernaufgabe

In [21]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [22]:
df_pc_eda

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
0,b694e2087c446e6905b08cd9360cfe45,bed_bath_table,1,delivered,2017-11-08 17:50:22,2017-11-09 17:48:09,99.99,36.21,5
1,af0a99476d96dcc1a1baa7c0d9ff6b9d,health_beauty,1,delivered,2018-07-26 20:32:46,2018-07-26 21:10:16,529.65,54.43,5
2,eb8c629f70275fd1c4f809116cce1efc,furniture_decor,2,delivered,2018-04-18 20:45:20,2018-04-18 21:11:09,35.00,17.97,4
3,2e830c73f28d3b8542af03cf2637dfc4,electronics,1,delivered,2018-03-02 09:59:12,2018-03-03 02:55:25,11.60,15.10,5
4,5411e9269501a870cabf632f05655131,stationery,1,delivered,2018-01-10 10:41:32,2018-01-12 02:35:33,129.00,18.15,1
...,...,...,...,...,...,...,...,...,...
100703,4ce9ab528124f89e091b17d11aa2e97c,computers_accessories,1,delivered,2018-03-29 15:09:18,2018-03-30 03:10:26,47.75,4.01,3
100704,4e1346d7b7e02c737a366b086462e33e,bed_bath_table,2,delivered,2018-08-05 15:42:51,2018-08-05 15:50:20,11.99,42.54,5
100705,50147c350ddbaff0646aeac455c7473e,bed_bath_table,1,delivered,2017-03-20 09:28:58,2017-03-20 09:28:58,119.90,9.60,<NA>
100706,928e52a9ad53a294fdcc91bcf59d1751,housewares,1,delivered,2018-06-07 17:52:27,2018-06-07 18:17:22,130.00,20.64,<NA>


In [23]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
count,100708.000000,100708,100695,100708.000000,100708.000000,99940.0
mean,1.103597,2018-01-01 16:12:59.141210,2018-01-02 03:32:58.710333,124.151016,20.142170,4.088883
min,1.000000,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000,1.0
25%,1.000000,2017-09-13 17:09:08,2017-09-14 02:45:40,40.140000,13.180000,4.0
50%,1.000000,2018-01-20 13:59:55.500000,2018-01-20 20:00:10,78.000000,16.360000,5.0
75%,1.000000,2018-05-05 21:22:12.750000,2018-05-06 13:50:11,139.000000,21.260000,5.0
max,20.000000,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.680000,5.0
std,0.461837,NaN,NaN,187.484910,15.898289,1.342309


In [24]:
df_pc_eda.dtypes

product_id                               object
product_category_name_english            object
order_count                               int64
order_status                             object
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
price                                   float64
freight_value                           float64
review_score                              Int64
dtype: object

In [25]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

product_id                            category
product_category_name_english         category
order_count                              Int16
order_status                          category
order_purchase_timestamp         datetime64[s]
order_approved_at                datetime64[s]
price                                  float32
freight_value                          float32
review_score                          category
dtype: object

In [26]:
df_pc_eda.describe()

,order_count,order_purchase_timestamp,order_approved_at,price,freight_value
count,100708.0,100708,100695,100708.000000,100708.000000
mean,1.103597,2018-01-01 16:12:59,2018-01-02 03:32:58,124.151009,20.142168
min,1.0,2016-09-04 21:15:19,2016-09-15 12:16:38,0.850000,0.000000
25%,1.0,2017-09-13 17:09:08,2017-09-14 02:45:40,40.139999,13.180000
50%,1.0,2018-01-20 13:59:55,2018-01-20 20:00:10,78.000000,16.360001
75%,1.0,2018-05-05 21:22:12,2018-05-06 13:50:11,139.000000,21.260000
max,20.0,2018-09-03 09:06:57,2018-09-03 17:40:06,6735.000000,409.679993
std,0.461837,NaN,NaN,187.484909,15.898289


In [27]:
df_pc_eda.isna().sum()

product_id                         0
product_category_name_english      0
order_count                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 13
price                              0
freight_value                      0
review_score                     768
dtype: int64

In [28]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

,product_id,product_category_name_english,order_count,order_status,order_purchase_timestamp,order_approved_at,price,freight_value,review_score
1602,e35b9be1fd37ec2d6e0be7ddf2d357b6,fashion_bags_accessories,1,delivered,2018-01-03 01:34:09,2018-01-03 01:49:03,24.900000,14.100000,NaN
1603,9007d9a8a0d332c61d9dd611fa341f4b,stationery,2,delivered,2018-04-09 13:11:55,2018-04-09 13:29:45,7.900000,8.290000,NaN
1604,bc7496ccc90a64c022a56021c65c9cda,housewares,1,delivered,2018-06-21 12:20:52,2018-06-21 12:41:03,19.900000,13.580000,NaN
1605,b931645cdc2d9868f01544e8db63f5ab,garden_tools,2,delivered,2017-03-13 13:33:07,2017-03-13 13:33:07,69.000000,14.250000,NaN
1606,04601b648d7d2dcae6e285a41e276a3f,toys,1,delivered,2017-11-29 20:44:54,2017-11-29 20:55:39,49.900002,16.600000,NaN
1607,827c4a77226a3e7ef259d61eaa775df9,furniture_decor,6,delivered,2018-01-03 21:20:26,2018-01-03 21:29:22,39.900002,12.480000,NaN
1608,dafe2eb26081f760c3eb2181e92c22ee,furniture_decor,2,delivered,2018-05-07 20:27:12,2018-05-07 20:51:25,40.000000,11.150000,NaN
1609,332b3e10a1e54aa72c52f41221eb53ab,cool_stuff,1,delivered,2017-11-27 10:27:16,2017-11-29 10:30:27,89.900002,13.650000,NaN
1610,87b08e712cc4c9fe70984c5a24b29e2f,toys,1,delivered,2017-07-17 21:25:23,2017-07-19 03:45:26,63.900002,16.889999,NaN
1611,4520766ec412348b8d4caa5e8a18c464,auto,1,delivered,2018-06-22 22:39:55,2018-06-23 01:40:00,27.490000,12.850000,NaN


In [29]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

Gesamte Duplikate: 0


In [30]:
con.execute("CREATE TABLE product_category AS SELECT * FROM df_pc_eda")

### EDA für dritte Kernaufgabe

In [31]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
        r.review_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp,
    oi.product_id,
    pcnt.product_category_name_english,
    Count(oi.product_id) AS product_count,                
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
WHERE o.order_status IN ('delivered')
Group BY o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
         o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
         r.review_id, r.review_score, r.review_comment_title, r.review_comment_message, 
                     r.review_creation_date, r.review_answer_timestamp,
         oi.product_id, s.seller_id, s.seller_city, s.seller_state, 
                     c.customer_city, c.customer_state, pcnt.product_category_name_english
    """)

In [32]:
df_service_eda

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
0,14282bc70be9bdda515182fb1ce62af4,delivered,2018-04-18 14:18:09,2018-04-19 02:52:02,2018-04-20 00:47:44,2018-04-26 16:26:38,2018-05-11,19af0595fd70818384dafe29191748d7,2,bom,...,2018-04-27 20:35:22,b05e00841a6dad404ef34ae67807879a,furniture_decor,2,aac29b1b99776be73c3049939652091d,uberlandia,MG,belo horizonte,MG,furniture_decor
1,c6ab8a5c6fbd3bf57e9ad8407d950cc6,delivered,2018-01-29 11:04:44,2018-01-29 11:21:15,2018-01-29 22:37:54,2018-01-30 20:37:59,2018-02-14,f6c49a0d64b4c1dbe189545fcae600cc,3,None,...,2018-01-31 23:09:42,e9b2560544c293e02f2b966041e10f24,sports_leisure,1,8e2b3afb420011ef0c88c9d5f11ea526,campinas,SP,sao paulo,SP,sports_leisure
2,b94f8c4cf21758ae29b895f3edc412d6,delivered,2018-01-02 16:06:06,2018-01-04 05:12:18,2018-01-08 11:53:10,2018-01-18 17:07:01,2018-01-30,85776fb05eaf4da541d91f32f130d312,5,None,...,2018-01-20 09:52:53,2c28b6fda526be32ed276b4dfc421acb,fashion_bags_accessories,1,e5a3438891c0bfdb9394643f95273d8e,limeira,SP,silvianopolis,MG,fashion_bags_accessories
3,a30cba7093267abeb0f8489d5c16c7af,delivered,2017-03-21 09:21:54,2017-03-21 09:21:54,2017-03-21 10:38:43,2017-03-27 13:56:19,2017-04-11,348f6373b7a445885d64a4d8dc9d5c21,1,None,...,2017-03-31 12:49:32,be62f0c24387eae79f18265ad2a5d8cc,sports_leisure,3,76d5af76d0271110f9af36c92573f765,sao paulo,SP,rio de janeiro,RJ,sports_leisure
4,d139bb03af910e29eab280ed63bec613,delivered,2018-08-23 11:50:26,2018-08-24 03:10:23,2018-08-24 15:07:00,2018-08-30 23:03:04,2018-09-13,1d9d7063154750120800386f85072f43,5,None,...,2018-09-01 04:24:34,c18ab1f0dc1937464d516ca610408e26,food_drink,2,e9779976487b77c6d4ac45f75ec7afe9,praia grande,SP,petropolis,RJ,food_drink
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98660,f726de924d4419207ba2ba5f81608a24,delivered,2017-10-29 13:42:54,2017-10-31 04:15:27,2017-10-31 16:43:42,2017-11-09 20:51:39,2017-11-28,b5da9b5254648a39bf9cd08375344f7e,5,None,...,2017-11-11 18:50:51,757172573ef8a2cc354243f3da4fa565,garden_tools,1,082e0bf4cb865a6533b1e8e498cc0255,botucatu,SP,paulo afonso,BA,garden_tools
98661,3ffd7375839be4547d1c419c90403c27,delivered,2017-11-25 21:39:12,2017-11-25 22:16:48,2017-11-30 12:02:53,2017-12-11 19:22:33,2017-12-19,ef4767983d3b56ce5eeefec96a50c488,3,None,...,2017-12-14 09:02:55,cf6514025ef5d3a93d69d2a7bfa4036b,bed_bath_table,1,d2374cbcbb3ca4ab1086534108cc3ab7,ibitinga,SP,praia grande,SP,bed_bath_table
98662,ffb18bf111fa70edf316eb0390427986,delivered,2017-11-27 13:29:05,2017-11-27 13:39:22,2017-11-28 22:15:05,2017-12-05 18:38:53,2017-12-21,2ff8c02dd6b39252a7af2801a9559ae6,5,None,...,2017-12-11 13:49:59,e86b81dcac341ea01df0260077cdf082,computers_accessories,1,a08692680c77d30a0b4280da5df01c5a,sao paulo,SP,jatai,GO,computers_accessories
98663,da737ef3ec9b9835b98345f6c155cdef,delivered,2018-08-11 23:32:03,2018-08-11 23:44:36,2018-08-13 16:09:00,2018-08-14 13:11:48,2018-08-16,0c8acc01f18cdaca0ecb03eb46a97f79,3,None,...,2018-08-19 16:23:49,57f2bc497c1a3ebe41ba7a06d78ed159,watches_gifts,1,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,ferraz de vasconcelos,SP,watches_gifts


In [33]:
df_service_eda.describe()


,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98652,98663,98657,98665,98665.000000,98665,98665,98665.000000
mean,2018-01-02 11:40:02.847706,2018-01-02 22:58:48.161487,2018-01-05 16:48:14.909003,2018-01-14 22:24:39.031310,2018-01-26 06:41:08.352505,4.127644,2018-01-14 17:11:58.613490,2018-01-17 20:45:37.395662,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.000000,2016-10-06 00:00:00,2016-10-07 18:32:28,1.000000
25%,2017-09-14 13:57:21,2017-09-14 22:25:18.250000,2017-09-18 19:05:10,2017-09-26 16:48:05,2017-10-05 00:00:00,4.000000,2017-09-27 00:00:00,2017-09-29 12:12:07,1.000000
50%,2018-01-21 13:28:36,2018-01-22 14:03:06,2018-01-24 18:52:39,2018-02-02 21:33:30,2018-02-16 00:00:00,5.000000,2018-02-03 00:00:00,2018-02-06 21:05:12,1.000000
75%,2018-05-06 18:49:55,2018-05-07 16:55:38.250000,2018-05-09 10:13:00,2018-05-16 17:24:34,2018-05-28 00:00:00,5.000000,2018-05-17 00:00:00,2018-05-20 20:13:32,1.000000
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.000000,2018-08-31 00:00:00,2018-10-29 12:27:35,20.000000
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [34]:
df_service_eda.dtypes


order_id                                   object
order_status                               object
order_purchase_timestamp           datetime64[us]
order_approved_at                  datetime64[us]
order_delivered_carrier_date       datetime64[us]
order_delivered_customer_date      datetime64[us]
order_estimated_delivery_date      datetime64[us]
review_id                                  object
review_score                                int64
review_comment_title                       object
review_comment_message                     object
review_creation_date               datetime64[us]
review_answer_timestamp            datetime64[us]
product_id                                 object
product_category_name_english              object
product_count                               int64
seller_id                                  object
seller_city                                object
seller_state                               object
customer_city                              object


In [35]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]',
              'review_id': 'category',
              'review_score': 'Int16',
              'review_comment_title': 'category',
              'review_comment_message': 'category',
              'review_creation_date': 'datetime64[s]',
              'review_answer_timestamp': 'datetime64[s]',
              'product_id': 'category',
              'product_category_name_english': 'category',
              'product_count': 'Int16',
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

order_id                                category
order_status                            category
order_purchase_timestamp           datetime64[s]
order_approved_at                  datetime64[s]
order_delivered_carrier_date       datetime64[s]
order_delivered_customer_date      datetime64[s]
order_estimated_delivery_date      datetime64[s]
review_id                               category
review_score                               Int16
review_comment_title                    category
review_comment_message                  category
review_creation_date               datetime64[s]
review_answer_timestamp            datetime64[s]
product_id                              category
product_category_name_english           category
product_count                              Int16
seller_id                               category
seller_city                             category
seller_state                            category
customer_city                           category
customer_state      

In [36]:
df_service_eda.isna().sum()

order_id                               0
order_status                           0
order_purchase_timestamp               0
order_approved_at                     13
order_delivered_carrier_date           2
order_delivered_customer_date          8
order_estimated_delivery_date          0
review_id                              0
review_score                           0
review_comment_title               86959
review_comment_message             58096
review_creation_date                   0
review_answer_timestamp                0
product_id                             0
product_category_name_english          0
product_count                          0
seller_id                              0
seller_city                            0
seller_state                           0
customer_city                          0
customer_state                         0
product_category_name_english_1        0
dtype: int64

In [37]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
4352,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30,bb311d9562ecbefc8e4be756d8999892,5,NaN,...,2018-07-10 11:38:13,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,sumare,SP,watches_gifts
12275,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16,c0dd6bec0375c376f044af102118526f,5,Entrega super rápida.,...,2018-06-29 16:26:37,2167c8f6252667c0eb9edd51520706a1,industry_commerce_and_business,1,0bb738e4d789e63e2267697c42d35a2d,sao roque,SP,quadra,SP,industry_commerce_and_business
36009,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18,f48c6c944a5d52dcca8ac5c4ec417cf2,5,NaN,...,2017-12-19 04:15:39,a50acd33ba7a8da8e9db65094fa990a4,auto,1,8581055ce74af1daba164fdbd55a40de,guarulhos,SP,cerquilho,SP,auto
50596,2aa91108853cecb43c84a5dc5b277475,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14,e945d1831a3d98008913fc31dcbb804d,5,NaN,...,2017-10-17 10:56:02,44c2baf621113fa7ac95fa06b4afbc68,furniture_decor,1,3f2af2670e104d1bcb54022274daeac5,terra boa,PR,indaiatuba,SP,furniture_decor
50973,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30,25e11638a3d01a87e8e62338a39eee28,5,NaN,...,2018-07-11 19:27:46,e7d5464b94c9a5963f7c686fc80145ad,watches_gifts,1,58f1a6197ed863543e0136bdedb3fce2,conselheiro lafaiete,MG,pindamonhangaba,SP,watches_gifts
58694,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19,d055795a562efffefe47ef81e5435322,5,Muito bom,...,2018-07-06 20:30:17,55bfa0307d7a46bed72c492259921231,books_general_interest,1,343e716476e3748b069f980efbaa294e,campinas,SP,ribeirao pires,SP,books_general_interest
64607,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23,4e755f114e50d33b9ac6a56e0d7d3ea9,5,NaN,...,2017-06-27 01:49:04,30b5b5635a79548a48d04162d971848f,sports_leisure,1,f9bbdd976532d50b7816d285a22bd01e,sao paulo,SP,porto alegre,RS,sports_leisure
81270,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24,ee2d30652e2f7fc00861074f795f5bf0,5,Excelente!,...,2018-07-07 18:48:09,ec165cd31c50585786ffda6feff5d0a6,toys,1,8bdd8e3fd58bafa48af76b2c5fd71974,sao paulo,SP,sao carlos,SP,toys
89118,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26,0d4c56af896dd6eb9de8edbaa1902d22,1,Péssimo,...,2018-06-16 13:55:00,a2a7efc985315e86d4f0f705701b342b,computers_accessories,1,ed4acab38528488b65a9a9c603ff024a,sao paulo,SP,guarulhos,SP,computers_accessories


In [38]:
df_service_eda['order_delivered_customer_date'] = df_service_eda['order_delivered_customer_date'].fillna(
    pd.to_datetime(df_service_eda['order_estimated_delivery_date'])
)

df_service_eda['order_delivered_carrier_date'] = df_service_eda['order_delivered_carrier_date'].fillna(
    pd.to_datetime(df_service_eda['order_approved_at']) + pd.Timedelta(days=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1


In [39]:
df_service_eda.describe()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_creation_date,review_answer_timestamp,product_count
count,98665,98665,98665,98665,98665,98665.0,98665,98665,98665.0
mean,2018-01-02 11:40:02,2018-01-02 21:57:29,2018-01-05 16:43:40,2018-01-14 22:37:27,2018-01-26 06:41:08,4.127644,2018-01-14 17:11:58,2018-01-17 20:45:37,1.099255
min,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-10-04 00:00:00,1.0,2016-10-06 00:00:00,2016-10-07 18:32:28,1.0
25%,2017-09-14 13:57:21,2017-09-14 21:45:17,2017-09-18 19:02:38,2017-09-26 16:48:13,2017-10-05 00:00:00,4.0,2017-09-27 00:00:00,2017-09-29 12:12:07,1.0
50%,2018-01-21 13:28:36,2018-01-22 14:00:56,2018-01-24 18:48:44,2018-02-02 21:39:55,2018-02-16 00:00:00,5.0,2018-02-03 00:00:00,2018-02-06 21:05:12,1.0
75%,2018-05-06 18:49:55,2018-05-07 16:53:20,2018-05-09 10:13:00,2018-05-16 17:29:13,2018-05-28 00:00:00,5.0,2018-05-17 00:00:00,2018-05-20 20:13:32,1.0
max,2018-08-29 15:00:37,2018-08-29 15:10:26,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-10-25 00:00:00,5.0,2018-08-31 00:00:00,2018-10-29 12:27:35,20.0
std,NaN,NaN,NaN,NaN,NaN,1.308843,NaN,NaN,0.451793


In [40]:
print("Gesamte Duplikate:", df_service_eda.duplicated().sum())


Gesamte Duplikate: 0


In [41]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_service_eda[(df_service_eda['order_id'] == order_id)]

order_data.head(63).sort_values('order_purchase_timestamp', ascending=True)

,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_id,review_score,review_comment_title,...,review_answer_timestamp,product_id,product_category_name_english,product_count,seller_id,seller_city,seller_state,customer_city,customer_state,product_category_name_english_1
9732,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,ebf9bc6cd600eadd681384e3116fda85,bed_bath_table,2,822166ed1e47908f7cfb49946d03c726,tres rios,RJ,sao paulo,SP,bed_bath_table
97343,895ab968e7bb0d5659d16cd74cd1650c,delivered,2017-08-08 20:26:31,2017-08-08 20:43:31,2017-08-10 11:58:14,2017-08-14 12:46:18,2017-08-30,eef5dbca8d37dfce6db7d7b16dd0525e,5,NaN,...,2017-08-17 22:17:55,5ddab10d5e0a23acb99acf56b62b3276,housewares,1,3d0cd21d41671c46f82cd11176bf7277,joinville,SC,sao paulo,SP,housewares


In [42]:
con.execute("CREATE TABLE service_analyse AS SELECT * FROM df_service_eda")

In [43]:
sql("SHOW TABLES")

,name
0,customer_rfm
1,customers
2,geolocation
3,order_items
4,order_payments
5,order_reviews
6,orders
7,product_category
8,product_category_name_translation
9,products
